Labo 8

In [36]:
import numpy as np

#EJ 1:
def trasponer(A):
    m = len(A)
    n = len(A[0])
    traspuesta = []
    for i in range(n):
        fila = []
        for j in range(m):
            fila.append(A[j][i])
        traspuesta.append(fila)
    return np.array(traspuesta)

def valorAbsoluto(n):
    if n<0:
        return n*(-1)
    return n

def multiplicar_matrices(A, B):

    res = []
    for filaA in A:
        f = []
        for c in range(B.shape[1]):  
            x = 0
            for i in range(len(filaA)):
                x += filaA[i] * B[i][c]
            f.append(x)
        res.append(f)
    return np.array(res)

def norma(x, p): 
    res = 0

    if p == "inf": 
        vals_abs = [valorAbsoluto(elem) for elem in x]
        return max(vals_abs)


    for elem in x: 
        elem_mod = valorAbsoluto(elem)
        elem_mod**=p
        res += elem_mod
    
    res**=(1/p)

    return res

def calcularAx(A,x):
    b = []
    for i in range(len(A)):
        suma = 0
        for j in range(len(A[0])):
            suma += A[i][j] * x[j]
        b += [suma]

    return np.array(b)

def producto_interno(u, v):
    res = 0
    for i in range(len(u)):
        res += u[i] * v[i]
    return np.array(res)

def normaMatInf(A): 
    
        fila_max= 0
        for fila in A:
             norma_fila = 0

             for elem in fila: 
                  norma_fila += valorAbsoluto(elem)

             if norma_fila > fila_max: 
                  fila_max = norma_fila

        return fila_max

def normaMatUno(A):
    return normaMatInf(trasponer(A))


def normaExacta(A, p=[1, 'inf']):
    if isinstance(p, int):
        if p == 1:
            return normaMatUno(A)
        else:
            return None
    elif isinstance(p, str):
        if p == "inf":
            return normaMatInf(A)
        else:
            return None
    else:
        if p[0] == 1:
            return [normaMatUno(A), normaMatInf(A)]
        else:
            return None


def producto_externo(u, v):
    n = len(u)
    m = len(v)
    M = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            M[i][j] = u[i] * v[j]
    return M

def fA(A, v):
   Av = calcularAx(A, v)
   w = 0

   if norma(Av, 2) != 0:
      w = Av / norma(Av, 2)

   return w

def fkA (A, v, K):
    k = 0
    while k<K: 
        v = fA (A, v)
        k += 1

    return v
     

#b 

def metpot2k(A,tol=1e-15,K=1000): 
    n = A.shape[0]
    v = np.random.uniform(0, 1, n)
    w = fkA(A, v, 2)
    e = producto_interno(w, v)
    k = 0

    while valorAbsoluto(e-1) > tol and k<K: 
        v = w
        w =fkA(A, v, 2)
        e = producto_interno(w, v)
        k += 1

    l = producto_interno(w, calcularAx(A, w))
    ep = e - 1

    return v, l, k

def recortar_matriz (A, n, m): 
    B = []

    for i in range (m-n): 
        v = np.zeros((m-n))
        for j in range(m-n): 
            v[j] = A[i + n][j+n]
        B.append(v)

    return np.array(B)

def diagRH(A,tol=1e-15,K=1000): 
    n = A.shape[0]
    m = A.shape[1]

    if n != m: 
        return None
    
    v1, l1, _ = metpot2k(A, tol, K)
    w = np.eye(n)[0] - v1
    Hv1 = np.eye(n) - 2* (producto_externo(w, w/(norma(w, 2)**2)))

    if n == 2: 
        S = Hv1
        D = multiplicar_matrices(Hv1, multiplicar_matrices(A, trasponer(Hv1)))
    else: 

        B =  multiplicar_matrices(Hv1, multiplicar_matrices(A, trasponer(Hv1)))
        C = recortar_matriz(B, 1, n)
        S2, D2 = diagRH(C, tol, K)
        D = np.zeros((n,n))
        D [0][0] =  l1
        for i in range(1, n): 
            for j in range(1,n): 
                D[i][j] = D2 [i-1][j-1]
        
        S3 = np.zeros((n,n))
        S3[0][0] =  1
        for i in range(1, n): 
            for j in range(1,n): 
                S3[i][j] = S2 [i-1][j-1]
        S = multiplicar_matrices(Hv1, S3)

    return S, D

def normaliza(X,p):
    n = len(X)
    y = []
    for i in range(n):
        m = len(X[i])
        fila_normalizada = np.zeros(m)
        for j in range(m):
            elemento = X[i][j]
            elemento_normalizado = elemento/norma(X[i],p)
            
            fila_normalizada[j] = elemento_normalizado
        y += [fila_normalizada]

    return y


In [37]:
#EJ 1

def svd_reducida_aux(A, tol=1e-15):
    A_t = trasponer(A)  
    H = multiplicar_matrices(A_t.tolist(),A)
    S, D = diagRH(H, tol, K=1000)

    S_t = trasponer(S.tolist())
    m = D.shape[0]
    n = D.shape[1]

    autovalores = []
    autovectores = []
    for i in range(m):
        if valorAbsoluto(D[i][i]) > tol :
            autovalores += [D[i][i]]
            autovectores += [S_t[i]]


    autoval_ordenados = []
    autovect_ordenados = []
    if len(autovalores) > 0:
        for i in range(len(autovalores)):
            max_autovalor = 0
            max_autovector = []

            if autovalores[i] > max_autovalor:
                max_autovalor = autovalores[i]
                max_autovector = autovectores[i]

        autoval_ordenados += [max_autovalor]
        autovect_ordenados += [max_autovector]
        autovalores.pop(i)
        autovectores.pop(i)

    V = np.array(autovect_ordenados)
    autoval_ordenados = np.array(autoval_ordenados)
    E = autoval_ordenados ** (1/2)
    r = autoval_ordenados.shape[0]

    B = multiplicar_matrices(A, V)
    B_t = trasponer(B.tolist())
    U_t = []

    for i in range(r):
        columna_normalizada = normaliza(B_t[i],2)
        U_t += [columna_normalizada]

    U = trasponer(U_t)

    return U, E, V
    

def svd_reducida(A, tol = 1e-15):
    m = A.shape[0]
    n = A .shape[1]

    if m >= n:
        U, E, V = svd_reducida_aux(A, tol)
    else:
        A_t = trasponer(A)
        V, E, U = svd_reducida_aux(A_t, tol)

    return U, E, V


In [38]:
#TEST

# Matrices al azar
def genera_matriz_para_test(m,n=2,tam_nucleo=0):
    if tam_nucleo == 0:
        A = np.random.random((m,n))
    else:
        A = np.random.random((m,tam_nucleo))
        A = np.hstack([A,A])
    return(A)

def test_svd_reducida_mn(A,tol=1e-15):
    m,n = A.shape
    hU,hS,hV = svd_reducida(A,tol=tol)
    nU,nS,nVT = np.linalg.svd(A)
    r = len(hS)+1
    assert np.all(np.abs(np.abs(np.diag(hU.T @ nU))-1)<10**r*tol), 'Revisar calculo de hat U en ' + str((m,n))
    assert np.all(np.abs(np.abs(np.diag(nVT @ hV))-1)<10**r*tol), 'Revisar calculo de hat V en ' + str((m,n))
    assert len(hS) == len(nS[np.abs(nS)>tol]), 'Hay cantidades distintas de valores singulares en ' + str((m,n))
    assert np.all(np.abs(hS-nS[np.abs(nS)>tol])<10**r*tol), 'Hay diferencias en los valores singulares en ' + str((m,n))

for m in [2,5,10,20]:
    for n in [2,5,10,20]:
        for _ in range(10):
            A = genera_matriz_para_test(m,n)
            test_svd_reducida_mn(A)


IndexError: index 1 is out of bounds for axis 0 with size 1